# NB25 버블의 계량검정 — 25장

『법률가를 위한 블록체인 — 기술과 경제학으로 읽기』의 장별 노트북입니다. 본문의 코드와 같습니다.

- 모든 코드는 원리를 눈으로 보기 위한 **장난감 구현**입니다. 실제 자산·실제 개인키·실제 거래에 쓰지 마십시오(실사용 금지).
- 외부 API를 부르지 않습니다. 네트워크 없이 실행됩니다. 동봉 스냅샷 데이터가 필요한 셀은 그 데이터를 셀 안에 적어 두었습니다.
- 표준 라이브러리 밖의 패키지는 `numpy` 하나만 씁니다(Python 3.13·numpy 2.5.4에서 실행·확인). 설치 셀은 없습니다 — 없으면 `pip install numpy==2.5.4`.
- 출력은 엮은이가 실행한 결과를 그대로 담아 두었습니다. 다시 실행하면 같은 값이 나와야 합니다(난수는 시드를 고정했습니다).

In [ ]:
# NB25-1: 합성 자료에 PWY/PSY 검정 — 폭발 구간을 심은 가짜 가격에 SADF·GSADF 를 계산한다 (방법론 시연. 실제 시세 자료는 쓰지 않는다)
import numpy as np

rng = np.random.default_rng(25)      # 시드 고정 — 같은 결과가 재현된다

def make_series(T=200, bubble=(100, 130), beta=1.04, sigma=1.0, collapse_to=None):
    """표류 없는 랜덤워크에 [bubble[0], bubble[1]) 구간만 y_t = beta·y_{t-1} + e_t 로 자라게 심고, 끝나면 심기 전 수준 근처로 떨어뜨린다."""
    y = np.empty(T); y[0] = 30.0
    e = rng.normal(0.0, sigma, T)
    for t in range(1, T):
        if bubble[0] <= t < bubble[1]:
            y[t] = beta * y[t - 1] + e[t]
        elif t == bubble[1]:
            y[t] = (collapse_to if collapse_to is not None else y[bubble[0] - 1]) + e[t]
        else:
            y[t] = y[t - 1] + e[t]
    return y

def adf_grid(y, min_win):
    """모든 창 [s, e] 에 대한 시차 0 ADF t-통계량(Δy_t 를 상수와 y_{t-1} 에 회귀, y_{t-1} 계수의 t 비율)을 한꺼번에 계산한다.
    돌려주는 값: (T−1)×(T−1) 배열 A, A[s, e] = 창이 관측 s+1 … e+1 의 Δy 를 쓰는 회귀의 t. 너무 짧은 창은 NaN."""
    x = y[:-1]; z = np.diff(y)                     # x_t = y_{t-1}, z_t = Δy_t, t = 1 … T−1
    n = len(z)
    def ps(a):                                    # 앞에 0 을 붙인 누적합 — 구간합을 뺄셈 하나로
        return np.concatenate(([0.0], np.cumsum(a)))
    Px, Pz, Pxx, Pxz, Pzz = ps(x), ps(z), ps(x * x), ps(x * z), ps(z * z)
    s = np.arange(n)[:, None]; e = np.arange(n)[None, :]
    cnt = (e - s + 1).astype(float)
    Sx = Px[e + 1] - Px[s]; Sz = Pz[e + 1] - Pz[s]
    Sxx = Pxx[e + 1] - Pxx[s]; Sxz = Pxz[e + 1] - Pxz[s]; Szz = Pzz[e + 1] - Pzz[s]
    with np.errstate(invalid="ignore", divide="ignore"):
        den = cnt * Sxx - Sx * Sx
        beta = (cnt * Sxz - Sx * Sz) / den
        alpha = (Sz - beta * Sx) / cnt
        rss = Szz - alpha * Sz - beta * Sxz
        s2 = rss / (cnt - 2)
        t = beta / np.sqrt(s2 * cnt / den)
    t[(cnt < min_win) | (e < s)] = np.nan
    return t

def sadf_gsadf(y, r0):
    T = len(y); w = int(np.floor(r0 * T))
    A = adf_grid(y, w)
    sadf = np.nanmax(A[0, :])                      # 시작 고정(0), 끝을 늘림
    bsadf = np.nanmax(A, axis=0)                   # 끝 r2 마다 시작을 전부 훑은 최댓값 — 날짜 찍기용
    gsadf = np.nanmax(bsadf)                       # 모든 (r1, r2) 의 최댓값
    return sadf, gsadf, bsadf, A

T = 200
r0 = 0.01 + 1.8 / np.sqrt(T)                       # PSY 의 최소 창 규칙 → 0.137 (T = 200)
y = make_series(T)
sadf, gsadf, bsadf, A = sadf_gsadf(y, r0)
PSY_T200 = {"SADF 95%": 1.40, "SADF 99%": 1.90, "GSADF 95%": 2.08, "GSADF 99%": 2.70}   # PSY (2015) Table 1, 유한표본, T=200, r0=0.137
print(f"합성 자료: T = {T}, 폭발 구간 t ∈ [100, 130), β = 1.04, 그 뒤 붕괴. 최소 창 r0 = {r0:.3f} → {int(np.floor(r0*T))} 관측")
print(f"SADF  = {sadf:.3f}   (PSY 표 1 임계값 95% {PSY_T200['SADF 95%']:.2f}, 99% {PSY_T200['SADF 99%']:.2f})")
print(f"GSADF = {gsadf:.3f}   (PSY 표 1 임계값 95% {PSY_T200['GSADF 95%']:.2f}, 99% {PSY_T200['GSADF 99%']:.2f})")
print(f"전체 표본 한 번의 ADF(시작 0, 끝 T) = {A[0, -1]:.3f}  ← 폭발 구간이 긴 단위근 구간에 묻혀 보이지 않는다")
peak = int(np.nanargmax(bsadf)) + 1
print(f"BSADF 가 최대인 끝점 t = {peak} (심은 구간의 끝 130 직전이면 잘 잡은 것)")

합성 자료: T = 200, 폭발 구간 t ∈ [100, 130), β = 1.04, 그 뒤 붕괴. 최소 창 r0 = 0.137 → 27 관측
SADF  = 11.918   (PSY 표 1 임계값 95% 1.40, 99% 1.90)
GSADF = 12.361   (PSY 표 1 임계값 95% 2.08, 99% 2.70)
전체 표본 한 번의 ADF(시작 0, 끝 T) = -2.466  ← 폭발 구간이 긴 단위근 구간에 묻혀 보이지 않는다
BSADF 가 최대인 끝점 t = 129 (심은 구간의 끝 130 직전이면 잘 잡은 것)


In [ ]:
# NB25-2: 몬테카를로 임계값과 날짜 찍기 — 귀무(랜덤워크)를 많이 만들어 GSADF 의 95%·99% 분위수와 BSADF 임계값 열을 얻고, 합성 자료에 날짜를 찍는다 (방법론 시연)
def mc_critical_values(T, r0, reps=2000, seed=2025):
    g = np.random.default_rng(seed)
    w = int(np.floor(r0 * T))
    gs = np.empty(reps); bs = np.empty((reps, T - 1))
    for i in range(reps):
        yn = 30.0 + np.cumsum(g.normal(0.0, 1.0, T))           # 귀무: 표류 없는 랜덤워크
        A = adf_grid(yn, w)
        bs[i] = np.nanmax(A, axis=0); gs[i] = np.nanmax(bs[i])
    return gs, bs

gs_null, bs_null = mc_critical_values(T, r0)
cv95, cv99 = np.quantile(gs_null, [0.95, 0.99])
print(f"귀무 랜덤워크 {len(gs_null)}회 → GSADF 임계값 95% {cv95:.2f}, 99% {cv99:.2f}   (PSY 표 1: 2.08, 2.70 — 이 셀은 반복 2,000회·시차 0·표류 없음, PSY 는 약한 표류 d=η=1)")
print(f"합성 자료의 GSADF {gsadf:.3f} → 95% 기준 {'기각(폭발 있음)' if gsadf > cv95 else '기각 못 함'}, 99% 기준 {'기각' if gsadf > cv99 else '기각 못 함'}")

# 날짜 찍기: 끝점 t 마다 귀무 BSADF_t 의 95% 분위수를 임계값으로 쓴다(PSY 는 ⌊T r2⌋ 관측 기준 SADF 임계값을 쓰며, 여기서는 같은 몬테카를로에서 직접 얻은 열로 대신한다)
bs_cv95 = np.nanquantile(bs_null, 0.95, axis=0)
LT = int(np.ceil(np.log(T)))                      # 최소 지속 log(T) ≈ 5.3 → 6 관측
flag = bsadf > bs_cv95
episodes, start = [], None
for t, f in enumerate(flag):
    if f and start is None: start = t
    if (not f or t == len(flag) - 1) and start is not None:
        end = t if not f else t + 1
        if end - start >= LT: episodes.append((start + 1, end + 1))   # +1: Δy 인덱스 → 원 시계열 시점
        start = None
print(f"임계값 열(끝점별 BSADF 95%)로 찍은 폭발 구간(최소 지속 {LT} 관측): {episodes}")
print(f"심은 구간: [100, 130). 시작은 몇 관측 늦게, 끝은 붕괴 직후에 잡히는 것이 보통이다 — 검정은 「자라는 중」을 보지 「자라기 시작했다」를 보지 못한다.")
print("심은 구간 밖에 잡힌 구간이 있으면 그것은 거짓 양성이다 — 끝점마다 5% 를 허용하면 170여 개 끝점 가운데 어딘가는 우연히 넘는다(올다니 외가 말한 다중성 문제).")

귀무 랜덤워크 2000회 → GSADF 임계값 95% 2.11, 99% 2.65   (PSY 표 1: 2.08, 2.70 — 이 셀은 반복 2,000회·시차 0·표류 없음, PSY 는 약한 표류 d=η=1)
합성 자료의 GSADF 12.361 → 95% 기준 기각(폭발 있음), 99% 기준 기각
임계값 열(끝점별 BSADF 95%)로 찍은 폭발 구간(최소 지속 6 관측): [(49, 63), (102, 130)]
심은 구간: [100, 130). 시작은 몇 관측 늦게, 끝은 붕괴 직후에 잡히는 것이 보통이다 — 검정은 「자라는 중」을 보지 「자라기 시작했다」를 보지 못한다.
심은 구간 밖에 잡힌 구간이 있으면 그것은 거짓 양성이다 — 끝점마다 5% 를 허용하면 170여 개 끝점 가운데 어딘가는 우연히 넘는다(올다니 외가 말한 다중성 문제).


In [ ]:
# NB25-3: 시변 변동성 아래의 크기 왜곡과 와일드 부트스트랩 — 버블이 없는데 분산만 커지는 귀무 자료에 교과서 임계값을 쓰면 얼마나 자주 「버블」이라 하는가 (방법론 시연)
def null_with_vol_break(T, g, s1=1.0, s2=3.0, brk=None):
    brk = T // 2 if brk is None else brk
    sig = np.where(np.arange(T) < brk, s1, s2)
    return 30.0 + np.cumsum(g.normal(0.0, 1.0, T) * sig)

g = np.random.default_rng(2526); reps = 400; w = int(np.floor(r0 * T))
rej_const = rej_break = 0
for i in range(reps):
    y0 = 30.0 + np.cumsum(g.normal(0.0, 1.0, T))                       # 분산 일정
    y1 = null_with_vol_break(T, g)                                      # 중간에 분산 9배(표준편차 3배)
    rej_const += np.nanmax(adf_grid(y0, w)) > cv95
    rej_break += np.nanmax(adf_grid(y1, w)) > cv95
print(f"귀무 자료 {reps}개씩, 몬테카를로 95% 임계값 {cv95:.2f} 사용 — 명목 크기 5%")
print(f"  분산 일정:            기각률 {rej_const / reps:.1%}")
print(f"  중간에 표준편차 3배:  기각률 {rej_break / reps:.1%}   ← 버블이 하나도 없는데 이만큼 「버블」이라 한다")

# 와일드 부트스트랩(하프너·필립스–시 방식의 뼈대): 관측 자료의 잔차에 ±1 또는 정규 가중을 곱해 다시 만든 귀무 표본에서 임계값을 얻는다 — 분산의 시간 변화가 그대로 보존된다
def wild_bootstrap_cv(y, w, B=199, seed=7):
    gb = np.random.default_rng(seed)
    resid = np.diff(y) - np.mean(np.diff(y))                            # 귀무(단위근) 아래의 잔차: Δy 에서 평균을 뺀 것
    stats = np.empty(B)
    for b in range(B):
        eb = resid * gb.normal(0.0, 1.0, len(resid))                    # 와일드: 각 잔차에 독립 N(0,1) 을 곱한다
        yb = y[0] + np.concatenate(([0.0], np.cumsum(eb)))
        stats[b] = np.nanmax(adf_grid(yb, w))
    return np.quantile(stats, 0.95)

# 교과서 임계값으로는 「버블」로 판정되는 귀무 자료를 하나 고른다(시드 0 부터 차례로 — 재현 가능)
for sd in range(200):
    y_break = null_with_vol_break(T, np.random.default_rng(sd))
    stat = np.nanmax(adf_grid(y_break, w))
    if stat > cv95: break
cv_wb = wild_bootstrap_cv(y_break, w)
print(f"\n분산이 변하는 귀무 자료 하나(시드 {sd}): GSADF {stat:.2f}. 교과서 임계값 {cv95:.2f} → {'기각' if stat > cv95 else '기각 못 함'}; 와일드 부트스트랩 95% 임계값 {cv_wb:.2f} → {'기각' if stat > cv_wb else '기각 못 함'}")
print("부트스트랩 임계값은 자료의 변동성 구조를 따라 움직인다. 같은 통계량이 한 임계값으로는 「버블」, 다른 임계값으로는 「아님」이 될 수 있다.")

귀무 자료 400개씩, 몬테카를로 95% 임계값 2.11 사용 — 명목 크기 5%
  분산 일정:            기각률 4.8%
  중간에 표준편차 3배:  기각률 34.0%   ← 버블이 하나도 없는데 이만큼 「버블」이라 한다

분산이 변하는 귀무 자료 하나(시드 0): GSADF 2.32. 교과서 임계값 2.11 → 기각; 와일드 부트스트랩 95% 임계값 3.45 → 기각 못 함
부트스트랩 임계값은 자료의 변동성 구조를 따라 움직인다. 같은 통계량이 한 임계값으로는 「버블」, 다른 임계값으로는 「아님」이 될 수 있다.
